# 오류분석 + OOF 스태킹 — 로컬 VS Code (LB 최고 0.74219 멤버)
**환경:** 로컬(M1, GPU 없음). 멤버 재학습 없음 — 기존 OOF/test 산출물만 로드해 진단+블렌드. 수 분~십수 분.
**경로:** 노트북=`notebooks/lee/`, 데이터·출력=`../../data/`(notebooks의 형제 data 폴더).
**질문:** "과적합하나"가 아니라 **"스태킹이 힐클라이밍 블렌드(가중평균)를 이기나"**.
**순서:** STEP1(오류분석)이 먼저 — 중간계에서 멤버가 갈리는지가 스태킹 가치를 미리 알려줌. 안 갈리면 STEP2 생략하고 "가중평균 동급"으로 닫음.
**원칙:** OOF만 · 단일시드 ±0.0002 잡음 · 결정지표 = 블렌드 OOF 증분 paired-bootstrap CI 0 제외.

## 0. 설정 + 멤버 로드 (y 정렬 assert)

In [1]:
import os, warnings
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata
warnings.filterwarnings("ignore")
DATA_DIR="../../data"      # notebooks/lee/ 기준 형제 data 폴더
def P(n):
    p=os.path.join(DATA_DIR,n); assert os.path.exists(p), f"{p} 없음 — DATA_DIR 확인"; return p
def runr(x): return rankdata(x)/len(x)

train=pd.read_csv(P("train.csv")); y=train["임신 성공 여부"].astype(int).values; N=len(y)
print(f"train {train.shape} | base_rate={y.mean():.4f}")

# LB 최고(0.74219) 조합 멤버 OOF — 실제 컬럼명 기준
def L(name, y_check=True):
    d=pd.read_csv(P(name))
    if y_check and "y" in d.columns:
        assert np.array_equal(d["y"].astype(int).values,y), f"{name} y 불일치 — fold 정렬 깨짐!"
    return d
op =L("oof_predictions.csv")     # oof_lgb/cat/xgb/lr + y
v3 =L("oof_v3_trees.csv")        # oof_v3_lgb/cat/xgb + y
v23=L("oof_v2v3_trees.csv")      # oof_v2v3_lgb/cat/xgb + y
lr =L("oof_lin_ratio.csv")       # oof_lin_ratio + y
nn =L("oof_nn.csv")              # oof_nn + y

# 0.74219 제출 조합: lgb:v2v3, xgb:v3, cat:v2v3, lin:ratio(=v2), nn:base
OOF={
  "lgb_v2v3": v23["oof_v2v3_lgb"].values,
  "xgb_v3":   v3["oof_v3_xgb"].values,
  "cat_v2v3": v23["oof_v2v3_cat"].values,
  "lin_v2":   lr["oof_lin_ratio"].values,    # 내부명 ratio = 팀명 v2
  "nn_base":  nn["oof_nn"].values,
}
MEMBERS=list(OOF)
assert all(len(v)==N for v in OOF.values()), "멤버 길이 불일치"
for m in MEMBERS:
    a=roc_auc_score(y,OOF[m]); assert a<0.999, f"{m} 라벨 오로드 의심"
    print(f"  {m}: OOF AUC={a:.5f}")

train (256351, 69) | base_rate=0.2583
  lgb_v2v3: OOF AUC=0.73965
  xgb_v3: OOF AUC=0.73972
  cat_v2v3: OOF AUC=0.73974
  lin_v2: OOF AUC=0.72029
  nn_base: OOF AUC=0.73751


## 0-1. 운영 블렌드(0.74219) 재구성 — 비교 baseline
힐클라이밍 가중으로 멤버 OOF를 재조합(그리드와 동일 방식). 이게 스태킹이 이겨야 할 대상.

In [2]:
def hill(d,yy,n=150):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get)
    ens=[b]; s=runr(d[b]).copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+runr(d[k]))/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+runr(d[cb])
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; cnt=Counter(best[0]); w={k:cnt.get(k,0)/len(best[0]) for k in nm}
    return w, sum(w[k]*runr(d[k]) for k in nm)
w_op,p_blend=hill(OOF,y)
print("운영 블렌드 재구성 AUC=",round(roc_auc_score(y,p_blend),5))
print("가중치:",{k:round(v,3) for k,v in w_op.items() if v>0})

운영 블렌드 재구성 AUC= 0.74067
가중치: {'lgb_v2v3': 0.281, 'xgb_v3': 0.209, 'cat_v2v3': 0.266, 'lin_v2': 0.029, 'nn_base': 0.216}


## STEP 1 — 오류분석 (스태킹이 이길지 먼저 가늠)
### 1a. 확률 분포 — LB 최고가 얼마나 확신에 차 있나

In [3]:
print("블렌드 확률 분포(1/10/25/50/75/90/99%):", np.percentile(p_blend,[1,10,25,50,75,90,99]).round(3))
print("극단(<0.05 or >0.6) 비율:", round(((p_blend<0.05)|(p_blend>0.6)).mean(),3))
mid=(p_blend>=0.2)&(p_blend<=0.5)
print("중간계(0.2~0.5) 비율:", round(mid.mean(),3), "| 그 구간 성공률:", round(y[mid].mean(),3))
print("→ 중간계 비율 크면 메타 작업영역 넓음. 극단만이면 스태킹 무용.")

블렌드 확률 분포(1/10/25/50/75/90/99%): [0.032 0.087 0.254 0.503 0.746 0.895 0.986]
극단(<0.05 or >0.6) 비율: 0.431
중간계(0.2~0.5) 비율: 0.3 | 그 구간 성공률: 0.195
→ 중간계 비율 크면 메타 작업영역 넓음. 극단만이면 스태킹 무용.


### 1b. 중간계 멤버 불일치 — ★스태킹 가치의 핵심 지표

In [4]:
R=pd.DataFrame({m:OOF[m] for m in MEMBERS})
def mean_offdiag(corr):
    v=corr.values; iu=np.triu_indices(len(v),1); return v[iu].mean()
rho_all=R.corr(method="spearman"); rho_mid=R[mid].corr(method="spearman")
print("전체 Spearman:\n", rho_all.round(3))
print("\n중간계(블렌드 0.2~0.5)만 Spearman:\n", rho_mid.round(3))
print(f"\n  전체 ρ평균={mean_offdiag(rho_all):.3f} | 중간계 ρ평균={mean_offdiag(rho_mid):.3f}")
disp=R[mid].apply(runr).std(axis=1)
print(f"  중간계 멤버 분산(랭크std) 평균={disp.mean():.4f} | >0.1 비율={(disp>0.1).mean():.3f}")
print("\n[판정] 중간계 ρ가 전체보다 눈에 띄게 낮거나 분산>0.1 비율 상당 → 스태킹 유망(STEP2).")
print("       중간계도 ρ0.99 유지 → 메타 할일 없음 → 가중평균 동급(STEP2 생략, '해보고 닫음').")

전체 Spearman:
           lgb_v2v3  xgb_v3  cat_v2v3  lin_v2  nn_base
lgb_v2v3     1.000   0.989     0.986   0.894    0.970
xgb_v3       0.989   1.000     0.989   0.900    0.971
cat_v2v3     0.986   0.989     1.000   0.902    0.972
lin_v2       0.894   0.900     0.902   1.000    0.901
nn_base      0.970   0.971     0.972   0.901    1.000

중간계(블렌드 0.2~0.5)만 Spearman:
           lgb_v2v3  xgb_v3  cat_v2v3  lin_v2  nn_base
lgb_v2v3     1.000   0.930     0.913   0.529    0.797
xgb_v3       0.930   1.000     0.931   0.553    0.801
cat_v2v3     0.913   0.931     1.000   0.561    0.799
lin_v2       0.529   0.553     0.561   1.000    0.516
nn_base      0.797   0.801     0.799   0.516    1.000

  전체 ρ평균=0.947 | 중간계 ρ평균=0.733
  중간계 멤버 분산(랭크std) 평균=0.1282 | >0.1 비율=0.560

[판정] 중간계 ρ가 전체보다 눈에 띄게 낮거나 분산>0.1 비율 상당 → 스태킹 유망(STEP2).
       중간계도 ρ0.99 유지 → 메타 할일 없음 → 가중평균 동급(STEP2 생략, '해보고 닫음').


### 1c. 오답 구조 (보너스 — 새 피처 단서)

In [5]:
pct=pd.Series(p_blend).rank(pct=True).values
fn=(y==1)&(pct<0.3); fp=(y==0)&(pct>0.7)
print(f"고확신 오답 — FN(성공인데 하위30%) {fn.sum()}건 | FP(실패인데 상위30%) {fp.sum()}건")
for col in ["시술 유형","난자 출처","배아 생성 주요 이유","단일 배아 이식 여부"]:
    if col in train.columns:
        print(f"\n[{col}]")
        print("  FN:", train.loc[fn,col].value_counts(normalize=True).round(3).head(4).to_dict())
        print("  FP:", train.loc[fp,col].value_counts(normalize=True).round(3).head(4).to_dict())
print("\n→ 특정 모달리티에 FN/FP 쏠리면 빠진 신호=향후 피처 단서. 비구조적이면 천장 확인.")

고확신 오답 — FN(성공인데 하위30%) 4419건 | FP(실패인데 상위30%) 42686건

[시술 유형]
  FN: {'IVF': 0.892, 'DI': 0.108}
  FP: {'IVF': 1.0, 'DI': 0.0}

[난자 출처]
  FN: {'본인 제공': 0.873, '알 수 없음': 0.108, '기증 제공': 0.019}
  FP: {'본인 제공': 0.915, '기증 제공': 0.085, '알 수 없음': 0.0}

[배아 생성 주요 이유]
  FN: {'현재 시술용': 0.996, '배아 저장용': 0.002, '기증용, 현재 시술용': 0.002, '배아 저장용, 현재 시술용': 0.001}
  FP: {'현재 시술용': 0.972, '기증용, 현재 시술용': 0.028, '배아 저장용, 현재 시술용': 0.0, '기증용, 배아 저장용, 현재 시술용': 0.0}

[단일 배아 이식 여부]
  FN: {0.0: 0.906, 1.0: 0.094}
  FP: {0.0: 0.565, 1.0: 0.435}

→ 특정 모달리티에 FN/FP 쏠리면 빠진 신호=향후 피처 단서. 비구조적이면 천장 확인.


## STEP 2 — OOF 스태킹 (STEP1에서 중간계 갈림 확인 시에만)
메타=로지스틱 고정(C=1), 입력=랭크정규화. 메타도 OOF로(누수 방지).

In [6]:
X_meta=np.column_stack([runr(OOF[m]) for m in MEMBERS])
skf=StratifiedKFold(5,shuffle=True,random_state=42); oof_stack=np.zeros(N)
for tr,va in skf.split(X_meta,y):
    lrm=LogisticRegression(C=1.0,max_iter=1000).fit(X_meta[tr],y[tr])
    oof_stack[va]=lrm.predict_proba(X_meta[va])[:,1]
print("스태킹 OOF AUC =", round(roc_auc_score(y,oof_stack),5))

스태킹 OOF AUC = 0.74058


### 2b. 결정 게이트 — vs 힐클라이밍 블렌드

In [7]:
a_st=roc_auc_score(y,oof_stack); a_bl=roc_auc_score(y,p_blend)
rng=np.random.default_rng(0); ds=np.empty(2000)
for i in range(2000):
    ix=rng.integers(0,N,N); ds[i]=roc_auc_score(y[ix],oof_stack[ix])-roc_auc_score(y[ix],p_blend[ix])
lo,hi=np.percentile(ds,[2.5,97.5])
print(f"스태킹 {a_st:.5f} vs 블렌드 {a_bl:.5f} | 증분 {a_st-a_bl:+.5f} CI[{lo:+.5f},{hi:+.5f}]",
      "→ 채택후보(LB)" if lo>0 else "→ 가중평균 못넘음(닫음)")

스태킹 0.74058 vs 블렌드 0.74067 | 증분 -0.00009 CI[-0.00016,-0.00003] → 가중평균 못넘음(닫음)


### 2c. 멀티시드 + test 산출 (게이트 통과 시만)

In [8]:
incs=[]
for s in (42,7,2024):
    sk=StratifiedKFold(5,shuffle=True,random_state=s); ostk=np.zeros(N)
    for tr,va in sk.split(X_meta,y):
        lrm=LogisticRegression(C=1.0,max_iter=1000).fit(X_meta[tr],y[tr]); ostk[va]=lrm.predict_proba(X_meta[va])[:,1]
    incs.append(roc_auc_score(y,ostk)-roc_auc_score(y,p_blend))
print("멀티시드 증분(vs 블렌드):",[round(i,5) for i in incs],"| 평균",round(np.mean(incs),5),"±",round(np.std(incs),5))
# 게이트+멀티시드 통과 시 test 산출
if lo>0 and np.mean(incs)>0 and abs(np.mean(incs))>np.std(incs):
    def LT(name,col):
        d=pd.read_csv(P(name)); return d[col].values
    try:
        Tst={"lgb_v2v3":LT("test_v2v3_trees.csv","v2v3_lgb"),"xgb_v3":LT("test_v3_trees.csv","v3_xgb"),
             "cat_v2v3":LT("test_v2v3_trees.csv","v2v3_cat"),"lin_v2":LT("test_lin_ratio.csv","lin_ratio"),
             "nn_base":LT("test_nn.csv","test_nn")}
        Xt=np.column_stack([runr(Tst[m]) for m in MEMBERS])
        lr_full=LogisticRegression(C=1.0,max_iter=1000).fit(X_meta,y)
        p_test=lr_full.predict_proba(Xt)[:,1]
        sub=pd.read_csv(P("sample_submission.csv")); pc=[c for c in sub.columns if c!="ID"][0]; sub[pc]=p_test
        sub.to_csv(os.path.join(DATA_DIR,"submission_stack.csv"),index=False)
        print("메타 계수:",dict(zip(MEMBERS,lr_full.coef_[0].round(3))))
        print("💾 ../../data/submission_stack.csv (게이트+멀티시드 통과 → LB 후보)")
    except Exception as e: print("test 산출 스킵:",e)
else:
    print("게이트 미통과 또는 멀티시드 불안정 → test 산출 안 함('가중평균 동급'으로 닫음)")
# 진단 저장
pd.DataFrame({"oof_stack":oof_stack,"p_blend":p_blend,"y":y}).to_csv(os.path.join(DATA_DIR,"oof_stack_diag.csv"),index=False)
print("💾 ../../data/oof_stack_diag.csv (스태킹·블렌드 OOF 진단)")

멀티시드 증분(vs 블렌드): [-9e-05, -8e-05, -9e-05] | 평균 -8e-05 ± 1e-05
게이트 미통과 또는 멀티시드 불안정 → test 산출 안 함('가중평균 동급'으로 닫음)
💾 ../../data/oof_stack_diag.csv (스태킹·블렌드 OOF 진단)
